# NB25 — Lexical-novelty dwell deviation per result

Regime: `[LAB]` only. Requires gaze-fixation data from AdSERP.

2026-04-15 retrospective: this notebook started as a lexical-novelty dwell-deviation analysis (Peter Dixon-Moses's Passing-vs-Rubbernecking framing). The intended finding was "residual dwell above/below the novelty-predicted baseline carries click-prediction signal the cursor features don't." That framing did not survive the multivariate test (K6 feature-ablation, K8 query-baseline): the residual is primarily an absolute-dwell proxy, not a pure novelty-deviation signal. What *did* show up — unexpectedly and robustly — is K7: the residual is phase-dependent in the same way the §4.4 cursor-side ablation is phase-dependent, with a 3.54× Survey-vs-post-Survey Spearman ratio at the OSEC fixation-5 boundary. That is the primary finding from this notebook, and it supports the §2.4 bounded-rationality / soft-constraints framing from an independent gaze-side measurement.

## Key Claims

| K-ID | Claim | Value | Source cell |
|---|---|---|---|
| K1 | Spearman ρ(cos-sim to centroid of previously-fixated results, fixation duration), non-first-fixation subset | −0.1011, p = 3.76 × 10⁻²⁷, n = 11,342 | Cell 7 |
| K2 | Spearman ρ(log-dwell residual, clicked), matched LAB subset | +0.2342, p = 2.50 × 10⁻¹²⁷, n = 10,215 | Cell 9 |
| K3 | LOSO concat AUC delta, M4 + residual − M4, click prediction | +0.0022 (0.8588 vs 0.8566) | Cell 10 |
| K4 | LOSO paired per-fold delta, 47 folds | +0.0014 ± 0.0064, M4+residual beats M4 in 26/47 folds | Cell 10 |
| K5a | Click-rate delta: rubbernecking (residual > 0) vs passing (residual ≤ 0) | +0.1391 (22.3 % vs 8.4 %, 2.65× ratio) | Cell 11 |
| K5b | NB22-deferred rate delta: rubbernecking vs passing | +0.3812 (83.2 % vs 45.1 %, 1.85× ratio) | Cell 11 |
| K6 | Feature-ablation: drop `dwell_in_proximity_ms` from M4, does residual recover the AUC loss? | No. M4 − dwell = 0.8331; M4 − dwell + residual = 0.8329. Net recovery = −0.0002. | Cell 12 |
| K7 | Phase-dependent residual-click Spearman (Survey-phase vs post-Survey, OSEC fixation-5 boundary) | Survey ρ = +0.0652 (p = 0.086, ns, n = 694); post-Survey ρ = +0.2307 (p = 5.2 × 10⁻¹²³, n = 10,180); ratio 3.54×. Matches §4.4 cursor-side ablation phase locality. | Cell 13 |
| K8a | Query-centroid baseline: Spearman ρ(query_sim, dwell) | +0.0146, p = 0.12 (ns) — query embedding does not predict dwell | Cell 14 |
| K8b | Query-centroid baseline: Spearman ρ(query_residual, click) | +0.2366, p = 5.4 × 10⁻¹³⁰; ratio to K2 centroid baseline = 1.01 (identical) | Cell 14 |
| K9 | Absolute-dwell phase-locality control: Spearman ρ(absolute_dwell, click) in Survey vs post-Survey; isolates whether K7's phase effect is residual-specific or simply reflects absolute-dwell phase-locality | _TBD_ | Cell 15 |

Primary interpretation (after full K-pass). The lexical-novelty baseline is empirically real (K1 ρ = −0.10 p < 10⁻²⁶) but weak; the residual is predominantly absolute-dwell variance (K6, K8b nearly identical to K2). The surprising and primary result is K7: the residual exhibits the same phase-locality as the §4.4 cursor-side ablation (Survey ρ ≈ 0; post-Survey ρ ≈ +0.23; 3.54× ratio). Two independent signals — cursor approach geometry and per-fixation gaze dwell — both localize the decision signal to the OSEC Evaluate+Commit phase. This is a per-fixation companion to the §4.4 cursor-side phase-locality finding and is consistent with the §2.4 bounded-rationality / soft-constraints framing.

K9 disambiguates K7 — is the phase-dependent residual a genuine gaze-dwell signal or is it absolute-dwell redundancy? If absolute-dwell Spearman ρ(dwell, click) shows the same phase locality as the residual, then K7 is just phase-dependent absolute dwell in disguise (matching K6+K8 pattern). If the residual's phase locality is *stronger* than absolute dwell's, then the novelty normalization is doing additional phase-specific work.

Do not hand-type values above — K1–K8 are executed output from the 2026-04-15 run. K9 fills in after Cell 15 executes. After final execution, run `python notebooks-v2/update_key_claims.py` to refresh the aggregate.

## Context and prior work

Motivation (Peter Dixon-Moses, 2026-04-15). A boundedly-rational reader's per-fixation dwell time should be partially predictable from the lexical novelty of the content they are fixating — predictable content reads fast, novel content reads slower (Reichle/Rayner E-Z Reader lineage). The *residual* — actual dwell minus predicted dwell — is the cognitive reallocation the soft constraints hypothesis (Gray, Sims, Fu & Schoelles 2006) predicts: users pay attention beyond what the surface content warrants when the task demands it (rubbernecking) or dismiss faster than warranted (passing). The original hypothesis was that the residual would be the signal, not absolute dwell.

Retrospective on the 2026-04-15 run. The original hypothesis did not survive the multivariate test. K6 (feature-ablation: drop `dwell_in_proximity_ms` from M4, check if residual recovers the lost AUC) and K8 (query-centroid baseline: swap the novelty source, check if residual changes) both show that the residual is predominantly an absolute-dwell proxy rather than a pure novelty-deviation signal. The cos-sim → dwell relationship is real (K1 ρ = −0.10, p < 10⁻²⁶) but too weak to dominate the residual's variance, so the residual ends up as log-dwell with a small correction applied. K6 confirms this: removing M4's cursor dwell costs 0.023 AUC and the residual recovers nothing.

What the run found instead. K7 is the unexpected finding: the residual is phase-dependent in the same way the §4.4 cursor-side ablation is phase-dependent. Splitting the fixation sequence at NB13's OSEC fixation-5 boundary and recomputing the residual separately within each phase yields ρ(residual, click) = +0.065 (ns, n = 694) in the Survey phase and ρ = +0.231 (p ≈ 10⁻¹²³, n = 10,180) in the post-Survey phase — a 3.54× ratio that matches the cursor-side phase dissociation almost exactly (§4.4: Survey M4 AUC 0.643, post-Survey 0.820). Two independent signals — cursor approach geometry and per-fixation gaze dwell — both localize the decision signal to the OSEC Evaluate+Commit phase. That dissociation supports the §2.4 bounded-rationality / soft-constraints framing from a second empirical angle.

Operationalization. Each result has a 1024-dim mxbai-embed-large embedding (pre-computed in `AdSERP/data/serp-embeddings.json`). For each trial, as the user's gaze visits results in sequence, we compute the centroid of embeddings for previously-fixated results and the cosine similarity of the next result's embedding to that centroid. Fitting `log(fixation_duration) ~ cos_sim_to_centroid` gives an expected-dwell baseline; the residual is the dwell deviation.

Prior work on this project.

- `notebooks-v2/09_difficulty.ipynb` — SERP-level gaze duration vs token novelty. This notebook extends the grain one level down to per-result.
- `docs/null-findings/priming-null-result.md` — four prior methodologies invalidated priming at the result-summary level. K1 DOES find a significant novelty → dwell relationship (ρ = −0.10, p < 10⁻²⁶) at the finer-grained embedding-centroid level, which is a positive result against the prior four nulls — but only as a univariate correlation; the click-prediction use case (K3/K4/K6) does not benefit over M4.
- NB22 four-class taxonomy — provides `gaze_regression_label` per (trial, result) for the deferred / eval-rejected split used in the downstream LOSO comparison.
- NB21 M4 feature set — the nine approach features the residual is compared against.
- `scripts/phase_restricted_ablation.py` — the §4.4 cursor-side ablation that K7 mirrors.

Outcome variables: click, deferred (NB22 ground truth), eval-rejected, not-approached.

In [1]:
# ── Imports and data paths ────────────────────────────────────────────
import json
import sys
from pathlib import Path
from collections import defaultdict

import numpy as np
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path("/Users/andyed/Documents/dev/attentional-foraging")
sys.path.insert(0, str(ROOT / "notebooks-v2"))
from data_loader import (
    load_fixations, get_trial_meta, result_band_tops,
    assign_fixation_to_position,
)

SERP_EMBEDDINGS = ROOT / "AdSERP/data/serp-embeddings.json"
LAB_RECORDS = ROOT / "AdSERP/data/cursor-approach-features.json"
REGRESSION_CACHE = ROOT / "scripts/output/approach_threshold_sensitivity/regression_labels_cache.json"
M4_FEATURES = [
    "min_dist", "mean_dist", "final_dist", "retreat_dist",
    "dwell_in_proximity_ms",
    "mean_approach_velocity", "max_approach_velocity",
    "direction_changes", "frac_decreasing",
]
print("imports ok")

imports ok


In [2]:
# ── Load per-result embeddings ───────────────────────────────────────
# Schema: {trial_id: [{position, title, snippet, text, embedding: [1024 floats]}, ...]}
with open(SERP_EMBEDDINGS) as f:
    serp_embeddings = json.load(f)
print(f"loaded embeddings for {len(serp_embeddings):,} trials")

# Build a flat (trial_id, position) -> embedding lookup
emb_lookup = {}
for tid, results in serp_embeddings.items():
    for r in results:
        if "embedding" in r:
            emb_lookup[(tid, r["position"])] = np.array(r["embedding"], dtype=np.float32)
print(f"(trial_id, position) -> embedding entries: {len(emb_lookup):,}")
print(f"embedding dim: {next(iter(emb_lookup.values())).shape[0]}")

loaded embeddings for 2,776 trials


(trial_id, position) -> embedding entries: 27,520
embedding dim: 1024


In [3]:
# ── Load LAB records and NB22 regression labels ──────────────────────
lab_records = json.load(open(LAB_RECORDS))
regression_labels = np.array(json.load(open(REGRESSION_CACHE)), dtype=bool)
print(f"lab records: {len(lab_records):,}")
print(f"regression labels (NB22 gaze_regression): {len(regression_labels):,}")

# Build per-record lookup for the M4 features
lab_index = {(r["trial_id"], r["position"]): i for i, r in enumerate(lab_records)}
print(f"indexed {len(lab_index):,} (trial, position) records")

lab records: 13,419
regression labels (NB22 gaze_regression): 13,419
indexed 13,419 (trial, position) records


In [4]:
# ── Per-result fixation duration aggregation ─────────────────────────
# For each trial, walk the fixation sequence and aggregate dwell time
# per result position. Also record the first-fixation timestamp per
# position so we can track fixation order within trial.

per_result_dwell = {}      # (trial_id, position) -> total fixation ms
per_result_first_t = {}    # (trial_id, position) -> ts of first fixation on that position
fixation_order = {}        # trial_id -> [positions in order of first fixation]
n_skipped = 0

trial_ids = sorted(set(tid for (tid, _) in emb_lookup))
for i, tid in enumerate(trial_ids):
    if i % 300 == 0:
        print(f"  {i}/{len(trial_ids)}  (skipped {n_skipped})")
    fixations = load_fixations(tid)
    if fixations is None or len(fixations) < 2:
        n_skipped += 1
        continue
    meta = get_trial_meta(tid)
    if meta is None:
        n_skipped += 1
        continue
    doc_h, _, _ = meta
    n_results = len([r for r in serp_embeddings.get(tid, []) if "embedding" in r])
    if n_results == 0:
        n_skipped += 1
        continue
    tops = result_band_tops(n_results, doc_h)

    seen_positions = []  # order of first fixation
    for fix in fixations:
        p = assign_fixation_to_position(fix["y"], tops, n_results)
        if p < 0:
            continue
        key = (tid, p)
        per_result_dwell[key] = per_result_dwell.get(key, 0.0) + float(fix.get("d", 0))
        if key not in per_result_first_t:
            per_result_first_t[key] = fix["t"]
            seen_positions.append(p)
    fixation_order[tid] = seen_positions

print(f"\nper-result dwell entries: {len(per_result_dwell):,}")
print(f"trials with fixation order: {len(fixation_order):,}")
print(f"skipped trials: {n_skipped}")

  0/2776  (skipped 0)


  300/2776  (skipped 0)


  600/2776  (skipped 0)


  900/2776  (skipped 0)


  1200/2776  (skipped 0)


  1500/2776  (skipped 0)


  1800/2776  (skipped 1)


  2100/2776  (skipped 1)


  2400/2776  (skipped 2)


  2700/2776  (skipped 2)

per-result dwell entries: 14,115
trials with fixation order: 2,774
skipped trials: 2


In [5]:
# ── Centroid tracking within trial ───────────────────────────────────
# For each (trial, position) pair, compute the centroid of embeddings for
# results fixated BEFORE this one in the trial's fixation sequence. Then
# compute cos-sim between this result's embedding and the centroid.
#
# First-fixated result has no prior centroid — we use the global mean of
# all embeddings as a prior (alternative: skip; we keep it to preserve
# sample size, and flag first-fixations for a secondary analysis).

global_mean_emb = np.mean(list(emb_lookup.values()), axis=0)
print(f"global mean embedding shape: {global_mean_emb.shape}")

def cosine_sim(a, b):
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

sim_to_centroid = {}  # (trial_id, position) -> cos-sim
is_first_fixation = {}  # (trial_id, position) -> bool

for tid, order in fixation_order.items():
    for i, pos in enumerate(order):
        key = (tid, pos)
        if key not in emb_lookup:
            continue
        if i == 0:
            centroid = global_mean_emb
            is_first_fixation[key] = True
        else:
            prior_embs = [emb_lookup[(tid, p)] for p in order[:i] if (tid, p) in emb_lookup]
            if not prior_embs:
                centroid = global_mean_emb
                is_first_fixation[key] = True
            else:
                centroid = np.mean(prior_embs, axis=0)
                is_first_fixation[key] = False
        sim_to_centroid[key] = cosine_sim(emb_lookup[key], centroid)

print(f"\ncos-sim entries: {len(sim_to_centroid):,}")
print(f"first-fixation entries (prior centroid fallback): {sum(is_first_fixation.values()):,}")

sims_arr = np.array(list(sim_to_centroid.values()))
print(f"cos-sim stats: mean={sims_arr.mean():.4f}, std={sims_arr.std():.4f}, "
      f"min={sims_arr.min():.4f}, max={sims_arr.max():.4f}")

global mean embedding shape: (1024,)

cos-sim entries: 14,115
first-fixation entries (prior centroid fallback): 2,773
cos-sim stats: mean=0.8123, std=0.1005, min=0.3668, max=1.0000


In [6]:
# ── K1: Spearman correlation between cos-sim-to-centroid and fixation duration ──
# Exclude first-fixation entries (prior centroid is a global-mean fallback,
# not a genuine within-trial novelty signal). Track the preserved key order
# so sim / dwell / residual arrays stay aligned downstream.

keys_valid = []
sim_vals = []
dwell_vals = []
for key, sim in sim_to_centroid.items():
    if is_first_fixation.get(key, False):
        continue
    if key not in per_result_dwell:
        continue
    dwell = per_result_dwell[key]
    if dwell <= 0:
        continue
    keys_valid.append(key)
    sim_vals.append(sim)
    dwell_vals.append(dwell)

sim_vals = np.array(sim_vals)
dwell_vals = np.array(dwell_vals)
print(f"n (non-first-fixation with dwell > 0): {len(sim_vals):,}")
print(f"keys_valid preserved: {len(keys_valid):,}  (must equal sim_vals length)")

rho, p = spearmanr(sim_vals, dwell_vals)
print(f"\n[K1] Spearman rho(cos_sim, fixation_duration) = {rho:+.4f}  (p = {p:.2e})")
print("  Interpretation:")
print("   - Negative rho: novel (low-sim) results get longer dwell (as predicted)")
print("   - Positive rho: predictable (high-sim) results get longer dwell (inverted)")
print("   - Near zero: no systematic relationship")


n (non-first-fixation with dwell > 0): 11,342
keys_valid preserved: 11,342  (must equal sim_vals length)

[K1] Spearman rho(cos_sim, fixation_duration) = -0.1011  (p = 3.76e-27)
  Interpretation:
   - Negative rho: novel (low-sim) results get longer dwell (as predicted)
   - Positive rho: predictable (high-sim) results get longer dwell (inverted)
   - Near zero: no systematic relationship


In [7]:
# ── Expected-dwell baseline model ───────────────────────────────────
# Fit a simple linear regression: log(fixation_duration) ~ cos_sim
# Log transform because fixation durations are right-skewed.
# Residual = log(actual) - log(predicted) is the dwell deviation variable.

log_dwell = np.log(dwell_vals)  # dwell_vals already filtered > 0 in Cell 7
X_sim = sim_vals.reshape(-1, 1)

reg = LinearRegression()
reg.fit(X_sim, log_dwell)
predicted_log_dwell = reg.predict(X_sim)
residuals = log_dwell - predicted_log_dwell

print(f"expected-dwell model: log(dwell) = {reg.coef_[0]:+.4f} * cos_sim + {reg.intercept_:.4f}")
print(f"residual stats: mean={residuals.mean():.4f}, std={residuals.std():.4f}")
print(f"  positive residual (rubbernecking) count: {(residuals > 0).sum():,}")
print(f"  negative residual (passing) count:       {(residuals < 0).sum():,}")

# Reattach residual to (trial_id, position) keys for downstream use.
# keys_valid was built in Cell 7 in the same iteration order as sim_vals /
# dwell_vals, so zip is order-safe here.
assert len(keys_valid) == len(residuals), (
    f"key/residual length mismatch: {len(keys_valid)} vs {len(residuals)}"
)
dwell_residual_by_key = dict(zip(keys_valid, residuals.tolist()))
print(f"\ndwell residual entries keyed by (trial, position): {len(dwell_residual_by_key):,}")


expected-dwell model: log(dwell) = -1.4372 * cos_sim + 8.6176
residual stats: mean=0.0000, std=1.1297
  positive residual (rubbernecking) count: 5,941
  negative residual (passing) count:       5,401

dwell residual entries keyed by (trial, position): 11,342


In [8]:
# ── K2: Residual vs click / deferred / eval-rejected outcomes ────────
# For each LAB record, pull the dwell residual (if present), the
# click flag, and the gaze_regression_label. Compute Spearman and
# group-mean comparisons.

rows = []
for i, r in enumerate(lab_records):
    key = (r["trial_id"], r["position"])
    if key not in dwell_residual_by_key:
        continue
    approached = float(r.get("min_dist", 9999)) < 100
    rows.append({
        "key": key,
        "residual": dwell_residual_by_key[key],
        "sim": sim_to_centroid[key],
        "was_clicked": bool(r.get("was_clicked", False)),
        "gaze_regression": bool(regression_labels[i]) if i < len(regression_labels) else False,
        "approached": approached,
        "position": r["position"],
        "participant": r["trial_id"].split("-")[0],
    })
print(f"matched records for downstream analysis: {len(rows):,}")

residuals_arr = np.array([row["residual"] for row in rows])
clicked_arr = np.array([row["was_clicked"] for row in rows], dtype=bool)

rho_click, p_click = spearmanr(residuals_arr, clicked_arr.astype(float))
print(f"\n[K2] Spearman rho(residual, clicked) = {rho_click:+.4f}  (p = {p_click:.2e})")

# Group means
print(f"\n  Mean residual by outcome:")
print(f"    clicked           : {residuals_arr[clicked_arr].mean():+.4f}  (n = {clicked_arr.sum():,})")
print(f"    non-clicked       : {residuals_arr[~clicked_arr].mean():+.4f}  (n = {(~clicked_arr).sum():,})")

matched records for downstream analysis: 10,215

[K2] Spearman rho(residual, clicked) = +0.2342  (p = 2.50e-127)

  Mean residual by outcome:
    clicked           : +0.5890  (n = 1,604)
    non-clicked       : -0.0991  (n = 8,611)


In [9]:
# ── K3, K4: LOSO comparison — M4 vs M4 + dwell_residual ──────────────
# Build feature matrices aligned on the intersection of LAB records and
# records with a dwell residual. Run LOSO logistic regression on click
# prediction for both feature sets and compare concat AUC + per-fold SD.

matched_idx = []
for row in rows:
    i = lab_index.get(row["key"])
    if i is None:
        continue
    matched_idx.append(i)
matched_idx = np.array(matched_idx)
print(f"matched to LAB index: {len(matched_idx):,}")

# M4 feature matrix on the matched subset
X_m4 = np.array([
    [float(lab_records[i].get(f, 0) or 0) for f in M4_FEATURES]
    for i in matched_idx
])
y = np.array([bool(lab_records[i].get("was_clicked", False)) for i in matched_idx], dtype=int)
groups = np.array([lab_records[i]["trial_id"].split("-")[0] for i in matched_idx])

# Augmented matrix = M4 + residual
residual_col = np.array([
    dwell_residual_by_key[(lab_records[i]["trial_id"], lab_records[i]["position"])]
    for i in matched_idx
]).reshape(-1, 1)
X_m4_plus = np.hstack([X_m4, residual_col])

print(f"\nfeature matrix shapes: M4 {X_m4.shape}, M4+residual {X_m4_plus.shape}")
print(f"click rate: {y.mean():.3f}")
print(f"n participants: {len(set(groups))}")

def loso_auc(X, y, groups, label):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=5000, class_weight="balanced", C=1.0)),
    ])
    gkf = GroupKFold(n_splits=len(set(groups)))
    y_proba = cross_val_predict(pipe, X, y, groups=groups, cv=gkf,
                                 method="predict_proba", n_jobs=1)[:, 1]
    concat_auc = float(roc_auc_score(y, y_proba))
    per_fold = []
    for _, test_idx in gkf.split(X, y, groups=groups):
        yt = y[test_idx]
        if len(np.unique(yt)) < 2:
            continue
        per_fold.append(float(roc_auc_score(yt, y_proba[test_idx])))
    per_fold_arr = np.array(per_fold)
    print(f"  {label}: concat AUC = {concat_auc:.4f}  "
          f"(per-fold {per_fold_arr.mean():.4f} ± {per_fold_arr.std(ddof=1):.4f}, "
          f"n_folds = {len(per_fold_arr)})")
    return concat_auc, per_fold_arr, y_proba

print("\n── LOSO click prediction ──")
auc_m4, folds_m4, _ = loso_auc(X_m4, y, groups, "M4 (9 features)")
auc_plus, folds_plus, _ = loso_auc(X_m4_plus, y, groups, "M4 + residual (10 features)")

print(f"\n[K3] concat AUC delta = {auc_plus - auc_m4:+.4f}")
if len(folds_m4) == len(folds_plus) and len(folds_m4) >= 2:
    paired = folds_plus - folds_m4
    print(f"[K4] paired per-fold delta: mean = {paired.mean():+.4f}, "
          f"sd = {paired.std(ddof=1):.4f}, "
          f"M4+residual beats M4 in {int((paired > 0).sum())}/{len(folds_m4)} folds")

matched to LAB index: 10,215

feature matrix shapes: M4 (10215, 9), M4+residual (10215, 10)
click rate: 0.157
n participants: 47

── LOSO click prediction ──


  M4 (9 features): concat AUC = 0.8566  (per-fold 0.8591 ± 0.0431, n_folds = 47)


  M4 + residual (10 features): concat AUC = 0.8588  (per-fold 0.8605 ± 0.0427, n_folds = 47)

[K3] concat AUC delta = +0.0022
[K4] paired per-fold delta: mean = +0.0014, sd = 0.0064, M4+residual beats M4 in 26/47 folds


In [10]:
# ── K5: Rubbernecking vs Passing (Peter's framing) ───────────────────
# Split the population by sign of residual and compare click / deferred /
# eval-rejected rates. Rubbernecking = positive residual (dwell longer
# than expected given novelty). Passing = negative residual.

rubbernecking = residuals_arr > 0
passing = residuals_arr <= 0

click_rub = clicked_arr[rubbernecking].mean()
click_pass = clicked_arr[passing].mean()
print(f"Rubbernecking (residual > 0): n={rubbernecking.sum():,}, click rate = {click_rub:.3f}")
print(f"Passing       (residual ≤ 0): n={passing.sum():,},   click rate = {click_pass:.3f}")
print(f"\n[K5] click-rate delta (rubbernecking − passing) = {click_rub - click_pass:+.4f}")

# Same comparison for gaze-regression (NB22 deferred label)
gaze_arr = np.array([row["gaze_regression"] for row in rows], dtype=bool)
print(f"\ngaze-regression (deferred) rates:")
print(f"  rubbernecking: {gaze_arr[rubbernecking].mean():.3f}")
print(f"  passing:       {gaze_arr[passing].mean():.3f}")
print(f"  delta:         {gaze_arr[rubbernecking].mean() - gaze_arr[passing].mean():+.4f}")

Rubbernecking (residual > 0): n=5,354, click rate = 0.223
Passing       (residual ≤ 0): n=4,861,   click rate = 0.084

[K5] click-rate delta (rubbernecking − passing) = +0.1391

gaze-regression (deferred) rates:
  rubbernecking: 0.832
  passing:       0.451
  delta:         +0.3812


In [11]:
# ── K6: Feature-ablation probe ────────────────────────────────────────
# Question: does the dwell residual recover click-prediction AUC that
# M4 loses when dwell_in_proximity_ms is dropped? If yes, residual is
# specifically a cognitive-dwell proxy and the M4 feature is its motor
# correlate. If no, the residual shares variance with multiple M4
# features, not just dwell.

dwell_idx = M4_FEATURES.index("dwell_in_proximity_ms")
print(f"dropping index {dwell_idx} ({M4_FEATURES[dwell_idx]}) from M4")

keep_idx = [j for j in range(len(M4_FEATURES)) if j != dwell_idx]
X_m4_nodwell = X_m4[:, keep_idx]
X_m4_nodwell_plus = np.hstack([X_m4_nodwell, residual_col])

print(f"\nfeature matrix shapes:")
print(f"  M4 − dwell             : {X_m4_nodwell.shape}")
print(f"  M4 − dwell + residual  : {X_m4_nodwell_plus.shape}")

print("\n── LOSO click prediction ──")
auc_nodwell, folds_nodwell, _ = loso_auc(X_m4_nodwell, y, groups, "M4 − dwell (8 features)")
auc_nodwell_plus, folds_nodwell_plus, _ = loso_auc(X_m4_nodwell_plus, y, groups, "M4 − dwell + residual (9 features)")

print(f"\nFor reference: M4 full (9 features) was {auc_m4:.4f}")
print(f"\n[K6] Recovery test:")
print(f"  AUC lost by dropping dwell:    {auc_m4 - auc_nodwell:+.4f}")
print(f"  AUC recovered by + residual:   {auc_nodwell_plus - auc_nodwell:+.4f}")
print(f"  Net (residual vs dwell):       {auc_nodwell_plus - auc_m4:+.4f}")

if auc_nodwell_plus >= auc_m4 - 0.005:
    print("\n  → Residual RECOVERS most or all of the dwell-feature loss.")
    print("    Interpretation: residual is a cognitive analogue of dwell_in_proximity_ms.")
elif auc_nodwell_plus >= auc_nodwell + 0.01:
    print("\n  → Residual recovers SOME of the dwell loss but not all.")
    print("    Interpretation: residual shares variance with dwell but also")
    print("    shares variance with other M4 features (velocity, direction_changes).")
else:
    print("\n  → Residual does NOT recover the dwell loss.")
    print("    Interpretation: residual variance is primarily captured by")
    print("    M4 features OTHER than dwell_in_proximity_ms.")


dropping index 4 (dwell_in_proximity_ms) from M4

feature matrix shapes:
  M4 − dwell             : (10215, 8)
  M4 − dwell + residual  : (10215, 9)

── LOSO click prediction ──


  M4 − dwell (8 features): concat AUC = 0.8331  (per-fold 0.8316 ± 0.0462, n_folds = 47)


  M4 − dwell + residual (9 features): concat AUC = 0.8329  (per-fold 0.8318 ± 0.0461, n_folds = 47)

For reference: M4 full (9 features) was 0.8566

[K6] Recovery test:
  AUC lost by dropping dwell:    +0.0234
  AUC recovered by + residual:   -0.0002
  Net (residual vs dwell):       -0.0236

  → Residual does NOT recover the dwell loss.
    Interpretation: residual variance is primarily captured by
    M4 features OTHER than dwell_in_proximity_ms.


In [12]:
# ── K7: Phase-dependent residual (Survey vs Evaluate) ────────────────
# Recompute per-result fixation duration splitting the fixation sequence
# by NB13's OSEC Survey → Evaluate boundary (end of fixation 5 = SURVEY_END).
# Fit separate expected-dwell baselines for each phase and compute the
# Spearman correlation between phase-specific residual and click outcome.
#
# Companion to §4.4 of the CIKM paper, which found that Survey-phase
# cursor collapses to position-only AUC (0.643) while post-Survey cursor
# preserves the whole-trial AUC (0.820). The per-fixation dwell residual
# should show the same phase locality if it is a semantic version of the
# same signal.

SURVEY_END = 5  # NB13 convention

per_result_dwell_survey = {}
per_result_dwell_post = {}

n_skipped_phase = 0
for tid in trial_ids:
    fixations = load_fixations(tid)
    if fixations is None or len(fixations) < SURVEY_END + 1:
        n_skipped_phase += 1
        continue
    meta = get_trial_meta(tid)
    if meta is None:
        n_skipped_phase += 1
        continue
    doc_h, _, _ = meta
    n_results = len([r for r in serp_embeddings.get(tid, []) if "embedding" in r])
    if n_results == 0:
        continue
    tops = result_band_tops(n_results, doc_h)
    # Boundary timestamp: end of 5th fixation
    boundary_t = int(fixations[SURVEY_END - 1]["t"] + fixations[SURVEY_END - 1].get("d", 0))

    for fix in fixations:
        p = assign_fixation_to_position(fix["y"], tops, n_results)
        if p < 0:
            continue
        key = (tid, p)
        dwell = float(fix.get("d", 0))
        if fix["t"] < boundary_t:
            per_result_dwell_survey[key] = per_result_dwell_survey.get(key, 0.0) + dwell
        else:
            per_result_dwell_post[key] = per_result_dwell_post.get(key, 0.0) + dwell

print(f"per-result Survey-phase dwell entries:      {len(per_result_dwell_survey):,}")
print(f"per-result post-Survey dwell entries:       {len(per_result_dwell_post):,}")
print(f"skipped trials (<{SURVEY_END + 1} fixations): {n_skipped_phase}")

def phase_residual_spearman(per_result_dwell_phase, label):
    """Fit expected-dwell baseline within a phase and Spearman vs click."""
    keys_p = []
    sims_p = []
    dwells_p = []
    for key, sim in sim_to_centroid.items():
        if is_first_fixation.get(key, False):
            continue
        if key not in per_result_dwell_phase:
            continue
        d = per_result_dwell_phase[key]
        if d <= 0:
            continue
        keys_p.append(key)
        sims_p.append(sim)
        dwells_p.append(d)
    sims_p = np.array(sims_p)
    dwells_p = np.array(dwells_p)
    if len(sims_p) < 100:
        print(f"\n  {label}: too few samples ({len(sims_p)}), skipping")
        return None
    # Expected-dwell baseline fit within this phase
    X_phase = sims_p.reshape(-1, 1)
    reg_phase = LinearRegression()
    reg_phase.fit(X_phase, np.log(dwells_p))
    resid_p = np.log(dwells_p) - reg_phase.predict(X_phase)
    # Match to click outcomes
    click_by_key = {(r["trial_id"], r["position"]): bool(r.get("was_clicked", False))
                    for r in lab_records}
    matched_resid = []
    matched_click = []
    for k, r in zip(keys_p, resid_p):
        if k in click_by_key:
            matched_resid.append(r)
            matched_click.append(click_by_key[k])
    matched_resid = np.array(matched_resid)
    matched_click = np.array(matched_click, dtype=float)
    if len(matched_resid) < 50:
        return None
    rho_p, p_p = spearmanr(matched_resid, matched_click)
    print(f"\n  {label}:")
    print(f"    n = {len(matched_resid):,}")
    print(f"    baseline slope = {reg_phase.coef_[0]:+.4f}")
    print(f"    Spearman ρ(residual, click) = {rho_p:+.4f}  (p = {p_p:.2e})")
    return {"rho": rho_p, "p": p_p, "n": len(matched_resid)}

print("\n── [K7] Phase-dependent residual-click correlation ──")
res_survey = phase_residual_spearman(per_result_dwell_survey, "Survey phase (t < fixation 5 end)")
res_post = phase_residual_spearman(per_result_dwell_post, "Post-Survey (t ≥ fixation 5 end)")

if res_survey and res_post:
    print(f"\n  Ratio: post-Survey / Survey = {res_post['rho'] / max(res_survey['rho'], 1e-6):.2f}")
    print("  §4.4 CIKM prediction: post-Survey cursor preserves the click signal;")
    print("  Survey-phase cursor collapses to noise. The dwell residual should show")
    print("  the same phase locality if it is the semantic twin of the cursor signal.")


per-result Survey-phase dwell entries:      3,390
per-result post-Survey dwell entries:       13,917
skipped trials (<6 fixations): 8

── [K7] Phase-dependent residual-click correlation ──

  Survey phase (t < fixation 5 end):
    n = 694
    baseline slope = +0.1042
    Spearman ρ(residual, click) = +0.0652  (p = 8.62e-02)

  Post-Survey (t ≥ fixation 5 end):
    n = 10,180
    baseline slope = -1.4630
    Spearman ρ(residual, click) = +0.2307  (p = 5.16e-123)

  Ratio: post-Survey / Survey = 3.54
  §4.4 CIKM prediction: post-Survey cursor preserves the click signal;
  Survey-phase cursor collapses to noise. The dwell residual should show
  the same phase locality if it is the semantic twin of the cursor signal.


In [13]:
# ── K8: Query-centroid baseline ──────────────────────────────────────
# Alternative novelty baseline: instead of the centroid of previously-
# fixated results, use the query embedding. Tests a different hypothesis:
# "how relevant is this result to the query" rather than "how novel is
# this result relative to what I've already read." The two baselines may
# capture different variance.

QUERY_EMBEDDINGS = ROOT / "AdSERP/data/query-embeddings.json"
with open(QUERY_EMBEDDINGS) as f:
    query_data = json.load(f)
print(f"loaded query embeddings for {len(query_data):,} trials")
# Schema: {trial_id: {"query": str, "embedding": [1024 floats]}}

query_emb_lookup = {
    tid: np.array(v["embedding"], dtype=np.float32)
    for tid, v in query_data.items()
    if isinstance(v, dict) and "embedding" in v
}
print(f"query embedding vectors: {len(query_emb_lookup):,}")
print(f"query embedding dim: {next(iter(query_emb_lookup.values())).shape[0]}")

# Compute cos_sim(result, query) for every (trial, position)
sim_to_query = {}
for (tid, pos), emb in emb_lookup.items():
    q_emb = query_emb_lookup.get(tid)
    if q_emb is None:
        continue
    sim_to_query[(tid, pos)] = cosine_sim(emb, q_emb)

print(f"\n(trial, position) -> query sim entries: {len(sim_to_query):,}")
sims_q_arr = np.array(list(sim_to_query.values()))
print(f"query cos-sim stats: mean={sims_q_arr.mean():.4f}, std={sims_q_arr.std():.4f}, "
      f"min={sims_q_arr.min():.4f}, max={sims_q_arr.max():.4f}")

# Align with dwell for the same (non-first-fixation, dwell>0) subset as cells 7-8
q_keys = []
q_sims = []
q_dwells = []
for key in keys_valid:
    if key not in sim_to_query:
        continue
    q_keys.append(key)
    q_sims.append(sim_to_query[key])
    q_dwells.append(per_result_dwell[key])
q_sims = np.array(q_sims)
q_dwells = np.array(q_dwells)
print(f"\nquery-baseline aligned sample: {len(q_sims):,}")

# K8a: Spearman between query sim and fixation duration
rho_q_dwell, p_q_dwell = spearmanr(q_sims, q_dwells)
print(f"\n[K8a] Spearman ρ(query_sim, dwell) = {rho_q_dwell:+.4f}  (p = {p_q_dwell:.2e})")

# Fit expected-dwell baseline on query-sim
reg_q = LinearRegression()
reg_q.fit(q_sims.reshape(-1, 1), np.log(q_dwells))
residuals_q = np.log(q_dwells) - reg_q.predict(q_sims.reshape(-1, 1))
print(f"query-baseline slope: {reg_q.coef_[0]:+.4f}")

# K8b: query-residual Spearman with click
click_by_key = {(r["trial_id"], r["position"]): bool(r.get("was_clicked", False))
                for r in lab_records}
q_matched_resid = []
q_matched_click = []
for k, r in zip(q_keys, residuals_q):
    if k in click_by_key:
        q_matched_resid.append(r)
        q_matched_click.append(click_by_key[k])
q_matched_resid = np.array(q_matched_resid)
q_matched_click = np.array(q_matched_click, dtype=float)

rho_q_click, p_q_click = spearmanr(q_matched_resid, q_matched_click)
print(f"\n[K8b] Spearman ρ(query_residual, click) = {rho_q_click:+.4f}  (p = {p_q_click:.2e})")
print(f"      n = {len(q_matched_resid):,}")
print(f"\nCompare: centroid-baseline residual-click ρ (K2) was +0.2342")
print(f"         query-baseline    residual-click ρ (K8b) = {rho_q_click:+.4f}")
print(f"         ratio: {abs(rho_q_click) / 0.2342:.2f}")


loaded query embeddings for 2,776 trials
query embedding vectors: 2,776
query embedding dim: 1024

(trial, position) -> query sim entries: 27,520
query cos-sim stats: mean=0.7976, std=0.0630, min=0.3028, max=0.9700

query-baseline aligned sample: 11,342

[K8a] Spearman ρ(query_sim, dwell) = +0.0146  (p = 1.21e-01)
query-baseline slope: +0.3754

[K8b] Spearman ρ(query_residual, click) = +0.2366  (p = 5.41e-130)
      n = 10,215

Compare: centroid-baseline residual-click ρ (K2) was +0.2342
         query-baseline    residual-click ρ (K8b) = +0.2366
         ratio: 1.01


In [14]:
# ── K9: Absolute-dwell phase-locality control ────────────────────────
# Question: is K7's phase-dependent residual-click correlation a
# GENUINELY residual-specific signal, or is it just absolute gaze dwell
# being phase-dependent and showing up through the residual?
#
# Design: compute Spearman rho(absolute_dwell, click) within each phase,
# same split as K7 (Survey first 5 fixations vs post-Survey). If the
# absolute-dwell phase locality matches K7's residual phase locality,
# then K7 is dwell-in-disguise (consistent with K6, K8). If the residual
# is meaningfully stronger, the novelty normalization adds phase-specific
# signal beyond raw dwell.

def phase_absolute_dwell_spearman(per_result_dwell_phase, label):
    click_by_key = {(r["trial_id"], r["position"]): bool(r.get("was_clicked", False))
                    for r in lab_records}
    matched_dwell = []
    matched_click = []
    for key, d in per_result_dwell_phase.items():
        if d <= 0:
            continue
        if key in click_by_key:
            matched_dwell.append(d)
            matched_click.append(click_by_key[key])
    matched_dwell = np.array(matched_dwell)
    matched_click = np.array(matched_click, dtype=float)
    if len(matched_dwell) < 50:
        print(f"  {label}: too few samples ({len(matched_dwell)}), skipping")
        return None
    rho_p, p_p = spearmanr(matched_dwell, matched_click)
    print(f"  {label}:")
    print(f"    n = {len(matched_dwell):,}")
    print(f"    Spearman ρ(absolute_dwell, click) = {rho_p:+.4f}  (p = {p_p:.2e})")
    return {"rho": rho_p, "p": p_p, "n": len(matched_dwell)}

print("── [K9] Absolute-dwell phase-locality control ──\n")
abs_survey = phase_absolute_dwell_spearman(per_result_dwell_survey, "Survey phase (absolute dwell)")
abs_post = phase_absolute_dwell_spearman(per_result_dwell_post, "Post-Survey (absolute dwell)")

print("\n── Comparison to K7 (residual-based phase locality) ──")
print("                          absolute dwell   residual")
if abs_survey and res_survey:
    print(f"  Survey phase  ρ      : {abs_survey['rho']:+.4f}          {res_survey['rho']:+.4f}")
if abs_post and res_post:
    print(f"  Post-Survey   ρ      : {abs_post['rho']:+.4f}          {res_post['rho']:+.4f}")
if abs_survey and abs_post:
    abs_ratio = abs_post['rho'] / max(abs_survey['rho'], 1e-6)
    print(f"  post/Survey ratio    : {abs_ratio:.2f}x          3.54x")

print("\n[K9] Interpretation:")
if abs_post and res_post:
    delta = res_post['rho'] - abs_post['rho']
    if abs(delta) < 0.02:
        print("  → Residual and absolute dwell have essentially identical")
        print("    phase-locality structure. K7 is driven by phase-dependent")
        print("    absolute dwell; the novelty normalization adds no marginal")
        print("    phase signal. Consistent with K6, K8 (residual ≈ dwell).")
    elif delta > 0.02:
        print(f"  → Residual phase locality exceeds absolute dwell by {delta:+.3f}.")
        print("    The novelty normalization adds phase-specific signal beyond")
        print("    raw dwell. K7 is residual-specific, not dwell redundant.")
    else:
        print(f"  → Absolute dwell phase locality exceeds residual by {-delta:+.3f}.")
        print("    The novelty normalization may be subtracting off useful")
        print("    phase-dependent signal. Unusual; investigate.")


── [K9] Absolute-dwell phase-locality control ──

  Survey phase (absolute dwell):
    n = 2,836
    Spearman ρ(absolute_dwell, click) = +0.0139  (p = 4.59e-01)
  Post-Survey (absolute dwell):
    n = 12,392
    Spearman ρ(absolute_dwell, click) = +0.2624  (p = 2.86e-194)

── Comparison to K7 (residual-based phase locality) ──
                          absolute dwell   residual
  Survey phase  ρ      : +0.0139          +0.0652
  Post-Survey   ρ      : +0.2624          +0.2307
  post/Survey ratio    : 18.86x          3.54x

[K9] Interpretation:
  → Absolute dwell phase locality exceeds residual by +0.032.
    The novelty normalization may be subtracting off useful
    phase-dependent signal. Unusual; investigate.


## Results summary

After executing all cells, transcribe the printed values into the Key Claims table at the top of this notebook. Then run:

```bash
python notebooks-v2/update_key_claims.py
```

to refresh `docs/notebook-key-claims.md`.

### Interpretation guide

- K1 negative (cos-sim negatively correlates with dwell): the baseline novelty-predicts-dwell assumption holds in AdSERP. The residual is a meaningful variable. Proceed to K2–K5.
- K1 near zero or positive: the embedding-centroid novelty model does not predict dwell in this data. The residual is uninterpretable as "deviation from novelty expectation." Consider (i) per-token novelty via word-level bounding boxes, (ii) a different baseline (e.g., cos-sim to the query rather than to the centroid of prior results), (iii) nonlinear expected-dwell model.
- K3/K4 positive and > per-fold SD: dwell residual adds click-prediction signal beyond the nine M4 cursor features. Would strengthen the §2.4 soft-constraints framing with a per-fixation test of the phase-dependence prediction.
- K3/K4 zero or negative: residual does not add signal. Consistent with the four prior priming nulls in `docs/null-findings/priming-null-result.md`. Note the specific methodological difference from those earlier tests (embedding-centroid baseline vs token-level priors) in any write-up.
- K5 positive (rubbernecking > passing click rate): dwell longer than the novelty model predicts is a click-positive signal — the user is over-attending to a result they eventually commit to. This is the Peter Dixon-Moses "Passing vs Rubbernecking" distinction operationalized.
- K5 zero or inverted: the rubbernecking/passing distinction does not track click outcomes directly. The distinction may still be informative for deferred vs eval-rejected (gaze-regression label in K2's secondary output).

### Follow-up work

- Token-level granularity. Requires word-level bounding boxes extracted from a headless-browser rendering of each SERP. E-Z Reader predicts per-token fixation durations from lexical properties; our result-level residual is a coarser instance.
- Query-based novelty baseline. Instead of centroid of prior fixated results, use cos-sim to the query embedding (from `query-embeddings.json`). This is a different hypothesis: *relevance to query*, not *novelty relative to what has been read*. May explain different variance.
- Phase-dependent residual. Compute the residual separately for Survey-phase (first 5 fixations) vs Evaluate-phase (post-fixation 5) cursor samples. §4.4 of the CIKM paper shows the click signal lives in Evaluate; the dwell-deviation signal may do the same.